# Start with standard imports

In [ ]:
try:
    import firedrake
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    import firedrake

In [ ]:
from firedrake import *
from firedrake.output import VTKFile

import numpy as np

import matplotlib.pyplot as plt
from matplotlib.animation import ArtistAnimation
from IPython.display import HTML

from tqdm.auto import tqdm

# Define the mesh and function spaces

In [ ]:
Lx = 0.3
Ly = 0.1
nx = 60
ny = 20

mesh = RectangleMesh(nx, ny, Lx, Ly, quadrilateral=True)

V = FunctionSpace(mesh, 'CG', 1)
W = VectorFunctionSpace(mesh, 'CG', 1)
DG0 = FunctionSpace(mesh, 'DG', 0)

output_file = VTKFile('Results.pvd')

output_deformation_factor = 50
coords = mesh.coordinates.dat.data.copy()

# Boundary conditions and Transient setup

In [ ]:
t_0 = 0
t_f = 1000
n_steps = 100
dt = (t_f - t_0) / n_steps

bc_th = DirichletBC(V, 1000, 2)

bc_mech = DirichletBC(W, Constant((0, 0)), 1)

# Material Properties and Functions

In [ ]:
# Thermal properties
k = Constant(10)
rho = Constant(8000)
c = Constant(500)

# Thermoelastic properties
alpha = Constant(1.2e-5)  # Thermal expansion coefficient
E = Constant(210e9)      # Young's Modulus (Pa)
nu = Constant(0.3)       # Poisson's ratio
T_ref = Constant(0)      # Reference temperature
mu = E / (2 * (1 + nu))
lmbda = E * nu / ((1 + nu) * (1 - 2 * nu))

T = Function(V, name='Temperature [C]')
u = Function(W, name='Displacement [m]')
von_mises_s = Function(DG0, name='von Mises Stress [Pa]')

# Heat Conduction Solver Setup

## Mathematical Derivation

### 1. Strong Form
The transient heat conduction equation (the Strong Form) over the domain $\Omega$ is defined as:
$$\rho c \frac{\partial T}{\partial t} - \nabla \cdot (k \nabla T) = 0$$

### 2. Time Discretisation (Backward Euler)
We discretise the time derivative using the **Backward Euler** scheme (implicit), which is unconditionally stable. We approximate the derivative at the current time step $n+1$ using the previous step $n$:
$$\frac{\partial T}{\partial t} \approx \frac{T^{n+1} - T^n}{\Delta t}$$

Substituting this into the strong form:
$$\rho c \left( \frac{T^{n+1} - T^n}{\Delta t} \right) - \nabla \cdot (k \nabla T^{n+1}) = 0$$

### 3. Weak Form Derivation
To obtain the variational (weak) form used by **Firedrake**, we multiply by a test function $v \in V$ and integrate over the domain $\Omega$:
$$\int_{\Omega} \rho c \left( \frac{T^{n+1} - T^n}{\Delta t} \right) v \, dx - \int_{\Omega} \nabla \cdot (k \nabla T^{n+1}) v \, dx = 0$$

Applying **integration by parts** (Divergence Theorem) to the diffusion term:
$$\int_{\Omega} \rho c \left( \frac{T^{n+1} - T^n}{\Delta t} \right) v \, dx + \int_{\Omega} k \nabla T^{n+1} \cdot \nabla v \, dx - \int_{\partial \Omega} (k \nabla T^{n+1} \cdot \mathbf{n}) v \, ds = 0$$



### 4. Final Variational Statement
Since we apply **Dirichlet boundary conditions** on specific boundaries, the test function $v$ is zero on those boundaries ($v=0$ on $\partial \Omega_D$). For the remaining boundaries, we assume no heat flux (insulation), so the boundary integral vanishes. Multiplying the entire equation by $\Delta t$, we get the form implemented in the code:
$$\int_{\Omega} \rho c (T^{n+1} - T^n) v \, dx + \Delta t \int_{\Omega} k \nabla T^{n+1} \cdot \nabla v \, dx = 0$$

In [ ]:
u_th = TrialFunction(V)
v_th = TestFunction(V)

T.assign(0)

thermal_form = (rho * c * inner(u_th-T, v_th) + dt * inner(k * grad(u_th), grad(v_th))) * dx

a_th = lhs(thermal_form)
L_th = rhs(thermal_form)

thermal_problem = LinearVariationalProblem(a_th, L_th, T, bcs=[bc_th], constant_jacobian=True)
thermal_solver = LinearVariationalSolver(thermal_problem)

# Thermoelastic Solver Setup

## Mathematical Derivation

### 1. Strong Form
In the absence of body forces and assuming quasi-static conditions (mechanical equilibrium is reached much faster than thermal diffusion), the balance of linear momentum is:
$$- \nabla \cdot \sigma = 0 \quad \text{in } \Omega$$

For a linear thermoelastic material, the stress tensor $\sigma$ is defined by the **Duhamel-Neumann constitutive law**:
$$\sigma = \lambda \text{tr}(\epsilon) \mathbf{I} + 2\mu \epsilon - \beta(T - T_{\text{ref}}) \mathbf{I}$$

Where:
* $\epsilon = \frac{1}{2}(\nabla \mathbf{u} + (\nabla \mathbf{u})^T)$ is the linearized strain tensor.
* $\lambda$ and $\mu$ are the **Lamé parameters**.
* $\beta = (3\lambda + 2\mu)\alpha$ is the thermal stress coefficient.
* $\alpha$ is the coefficient of linear thermal expansion.
* $T_{\text{ref}}$ is the reference temperature at which the material is stress-free.

### 2. Weak Form Derivation
To find the variational (weak) form, we multiply the strong form by a vector-valued test function $\mathbf{v} \in W$ and integrate over the domain $\Omega$:
$$\int_{\Omega} (- \nabla \cdot \sigma) \cdot \mathbf{v} \, dx = 0$$

Applying **integration by parts** (the Principle of Virtual Work):
$$\int_{\Omega} \sigma : \nabla \mathbf{v} \, dx - \int_{\partial \Omega} (\sigma \cdot \mathbf{n}) \cdot \mathbf{v} \, ds = 0$$

Since the stress tensor $\sigma$ is symmetric, the inner product with the gradient of the test function $\nabla \mathbf{v}$ is equivalent to the inner product with the symmetric strain operator $\epsilon(\mathbf{v})$:
$$\sigma : \nabla \mathbf{v} = \sigma : \epsilon(\mathbf{v})$$



### 3. Final Variational Statement
Assuming zero traction (Neumann) on the free boundaries and fixed (Dirichlet) conditions where specified, the boundary integral vanishes. The problem becomes: Find $\mathbf{u} \in W$ such that:
$$\int_{\Omega} \sigma(\mathbf{u}, T) : \epsilon(\mathbf{v}) \, dx = 0 \quad \forall \mathbf{v} \in W$$

In the Firedrake implementation, this is written as:
`mech_form = inner(sigma(u_mech, T), epsilon(v_mech)) * dx`

In [ ]:
u_mech = TrialFunction(W)
v_mech = TestFunction(W)

def epsilon(u):
    return 0.5 * (grad(u) + grad(u).T)

def sigma(u, T):
    thermal_strain = (3*lmbda + 2*mu) * alpha * (T - T_ref)
    return lmbda * div(u) * Identity(2) + 2*mu * epsilon(u) - thermal_strain * Identity(2)

# Mechanical variational problem
mech_form = inner(sigma(u_mech, T), epsilon(v_mech)) * dx

a_mech = lhs(mech_form)
L_mech = rhs(mech_form)

# Setup Solver
mech_prob = LinearVariationalProblem(a_mech, L_mech, u, bcs=[bc_mech], constant_jacobian=True)
mech_solver = LinearVariationalSolver(mech_prob)

# Animation setup

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 4))

offset = Ly/2
for ax, title in zip([ax1, ax2], ["Temperature Evolution", "Deformed von Mises Stress"]):
    ax.set_xlim(0, Lx + 2*offset)
    ax.set_ylim(-offset, Ly + offset)
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")

frames = []

# Setup Colorbars
# Temperature colorbar
tp = tripcolor(T, axes=ax1, vmin=0, vmax=1000, cmap='inferno')
cb1 = plt.colorbar(tp, ax=ax1)
cb1.set_label("Temperature [C]")

# von Mises colorbar (adjust vmax based on expected stress levels)
sp = tripcolor(von_mises_s, axes=ax2, vmin=0, vmax=5e8, cmap='coolwarm')
cb2 = plt.colorbar(sp, ax=ax2)
cb2.set_label("von Mises Stress [Pa]")

plt.tight_layout()
plt.close()

# Solver Loop

In [ ]:
for i in tqdm(range(n_steps), desc="Solving Transient System"):
    thermal_solver.solve()
    mech_solver.solve()

    s = sigma(u, T)
    s_xx, s_yy = s[0,0], s[1,1]
    s_xy = s[0,1]
    von_mises_s.project(sqrt(s_xx**2 + s_yy**2 - s_xx*s_yy + 3*s_xy**2))

    output_file.write(T, u, von_mises_s)

    mesh.coordinates.dat.data[:] = coords + u.dat.data[:]*output_deformation_factor
    c1 = tripcolor(T, axes=ax1, vmin=0, vmax=1000, cmap='inferno')
    c2 = tripcolor(von_mises_s, axes=ax2, vmin=0, vmax=5e8, cmap='coolwarm')
    frames.append([c1, c2])
    mesh.coordinates.dat.data[:] = coords

# Generate the animation

In [ ]:
plt.close()
ani = ArtistAnimation(fig, frames, interval=50, blit=True, repeat_delay=1000)
# HTML(ani.to_jshtml())
HTML(f'<div style="width:100%;">{ani.to_jshtml()}</div>')